In [1]:
# Import Libraries
import pandas as pd

# Load Data
df = pd.read_csv("../data/raw/train.csv")

In [2]:
# Convert Data-types
df['date'] = pd.to_datetime(df['date'])

df = df.sort_values(by=['store', 'item', 'date'])

df = df.reset_index(drop=True)

In [3]:
# Create Time Features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['day_of_month'] = df['date'].dt.day

In [4]:
# Create Lag Features (Historical Memory)
for lag in [1, 7, 14, 28, 56, 84]:
    df[f'lag_{lag}'] = df.groupby(['store', 'item'])['sales'].shift(lag)

In [5]:
# Create Rolling Features (Trends)
df['rolling_mean_7'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_std_7'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(7).std())

In [6]:
# Additional Rolling Features
df['rolling_mean_14'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(14).mean())

df['rolling_mean_28'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(28).mean())

# Rolling max (captures spikes)
df['rolling_max_7'] = df.groupby(['store', 'item'])['sales'].transform(lambda x: x.shift(1).rolling(7).max())

In [7]:
# Momentum Feature
df['diff_1'] = df['lag_1'] - df['lag_7']

In [8]:
# Drop Missing Values
df = df.dropna()

In [9]:
# Check Modified Data
print("Data Shape:", df.shape)
print("\n------------------------------\n")
df.head()

Data Shape: (871000, 20)

------------------------------



,date,store,item,sales,day_of_week,month,year,day_of_month,lag_1,lag_7,lag_14,lag_28,lag_56,lag_84,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_mean_28,rolling_max_7,diff_1
84,2013-03-26,1,1,16,1,3,2013,26,13.0,19.0,14.0,9.0,6.0,13.0,17.714286,3.199702,15.571429,14.607143,21.0,-6.0
85,2013-03-27,1,1,11,2,3,2013,27,16.0,14.0,13.0,9.0,9.0,11.0,17.285714,3.199702,15.714286,14.857143,21.0,2.0
86,2013-03-28,1,1,13,3,3,2013,28,11.0,17.0,10.0,10.0,13.0,14.0,16.857143,3.848314,15.571429,14.928571,21.0,-6.0
87,2013-03-29,1,1,17,4,3,2013,29,13.0,21.0,14.0,15.0,11.0,13.0,16.285714,4.111540,15.785714,15.035714,21.0,-8.0
88,2013-03-30,1,1,19,5,3,2013,30,17.0,21.0,10.0,13.0,21.0,10.0,15.714286,3.592320,16.000000,15.107143,21.0,-4.0


In [10]:
df.to_csv("../data/processed/feature_data.csv", index=False)